# Lakehouse-Einstieg: Datenkrake-Telemetrie in MinIO + Nessie + Iceberg

Dieses Notebook zeigt den Unterschied zwischen den bisherigen Speicherorten
(MariaDB, InfluxDB) und einem **Data Lakehouse**:

| | MariaDB / InfluxDB | Lakehouse (hier) |
|---|---|---|
| Speicherort | eigene Datenbank-Engine | offene Dateien (Parquet) in Objektspeicher (MinIO) |
| Zugriff | nur über die jeweilige DB | von jedem Tool, das Iceberg spricht (Spark, Trino, DuckDB, ...) |
| Versionierung | keine eingebaute | **Nessie**: Branches/Commits wie bei Git, auf ganzen Tabellen |
| Stärke | operativer Betrieb | große Datenmengen, Analyse, Data-Science-Workflows |

Ablauf in diesem Notebook:
1. Verbindung zu Spark + Nessie (Iceberg-REST-Katalog) aufbauen
2. Telemetriedaten aus der MariaDB der Datenkrake laden
3. Als Iceberg-Tabelle im Lakehouse ablegen
4. Einen Nessie-**Branch** anlegen und eine Änderung isoliert testen - ohne
   die Haupt-Tabelle zu berühren, genau wie ein Feature-Branch in Git

**Wichtig:** dieses Notebook läuft NICHT auf dem Raspberry Pi, sondern in
diesem separaten Data-Lake-Stack. Es braucht nur Netzzugriff auf die
Datenkrake-MariaDB (Standard: `datenkrake.local:3306`).

## 1. Spark-Session mit Nessie als Iceberg-REST-Katalog

Die Paket-Versionen (`nessie-spark-extensions-3.5_2.12`,
`iceberg-spark-runtime-3.5_2.12`) müssen zur im Container installierten
Spark-Version passen. Zelle darunter prüft das automatisch.

In [ ]:
import pyspark
print("Spark-Version im Container:", pyspark.__version__)
# Erwartet: 3.5.x - falls nicht, unten im packages-String "3.5" anpassen
# (z.B. auf "3.4" oder "3.6") und die passende Iceberg/Nessie-Version dazu
# suchen: https://projectnessie.org/guides/spark-s3/

In [ ]:
from pyspark.sql import SparkSession

NESSIE_URI = "http://nessie:19120/iceberg/main/"  # innerhalb des Docker-Netzes

spark = (
    SparkSession.builder
    .appName("Datenkrake Lakehouse Demo")
    .config("spark.jars.packages",
            "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.108.2,"
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.11.0,"
            "org.apache.iceberg:iceberg-aws-bundle:1.11.0")
    .config("spark.sql.extensions",
            "org.projectnessie.spark.extensions.NessieSparkSessionExtensions,"
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.nessie.type", "rest")
    .config("spark.sql.catalog.nessie.uri", NESSIE_URI)
    .config("spark.sql.catalogImplementation", "in-memory")
    .getOrCreate()
)
spark.sql("USE nessie")
print("Verbunden. Erste Zelle dauert etwas, da die Pakete heruntergeladen werden.")

## 2. Telemetriedaten aus der Datenkrake-MariaDB laden

Läuft bewusst über `pandas`/`pymysql` statt über einen JDBC-Treiber - das
spart eine zusätzliche, versionssensitive Abhängigkeit. Für sehr große
Datenmengen wäre ein JDBC-Connector performanter (siehe Hinweis am
Notebook-Ende).

In [ ]:
import pandas as pd
import pymysql

DB_HOST = "datenkrake.local"   # ggf. auf eure Pi-Adresse/IP anpassen
DB_USER = "mcp_read"
DB_PASSWORD = "changeMeMcp"
DB_NAME = "telemetry"

conn = pymysql.connect(host=DB_HOST, user=DB_USER, password=DB_PASSWORD, database=DB_NAME)
pdf = pd.read_sql(
    "SELECT id, ts, label, peak_freq, peak_db, sample_rate FROM audio_spectrum ORDER BY ts",
    conn,
)
conn.close()
print(f"{len(pdf)} Zeilen geladen.")
pdf.head()

## 3. Als Iceberg-Tabelle im Lakehouse ablegen

Der `CREATE TABLE ... USING iceberg` läuft auf dem `nessie`-Katalog aus
Schritt 1 - die Datei landet als Parquet in MinIO, die Metadaten dazu
verwaltet Nessie.

In [ ]:
sdf = spark.createDataFrame(pdf.astype({"ts": "str"}))
sdf.createOrReplaceTempView("audio_spectrum_import")

spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.telemetrie")
spark.sql("""DROP TABLE IF EXISTS nessie.telemetrie.audio_spectrum""")
spark.sql("""CREATE TABLE nessie.telemetrie.audio_spectrum USING iceberg AS
   SELECT * FROM audio_spectrum_import""")

spark.sql("SELECT COUNT(*) AS anzahl FROM nessie.telemetrie.audio_spectrum").show()

## 4. Nessie-Branch: isoliert experimentieren, ohne die Haupttabelle zu ändern

Das ist der Kernunterschied zu MariaDB/InfluxDB: Tabellen lassen sich wie
Dateien in Git branchen. Wir legen einen Branch `analyse-schlecht-only` an,
löschen dort probeweise alle `"gut"`-Datensätze - und zeigen, dass die
Haupttabelle (`main`) davon unberührt bleibt.

In [ ]:
spark.sql("CREATE BRANCH IF NOT EXISTS `analyse-schlecht-only` IN nessie FROM main")
spark.sql("USE REFERENCE `analyse-schlecht-only` IN nessie")

spark.sql("DELETE FROM nessie.telemetrie.audio_spectrum WHERE label = 'gut'")
print("Auf dem Branch verbleibend:")
spark.sql("SELECT label, COUNT(*) FROM nessie.telemetrie.audio_spectrum GROUP BY label").show()

spark.sql("USE REFERENCE main IN nessie")
print("Auf main unverändert:")
spark.sql("SELECT label, COUNT(*) FROM nessie.telemetrie.audio_spectrum GROUP BY label").show()

## Weiterführend

- **Web-UI von MinIO** (`http://localhost:9001`, Login siehe
  `docker-compose.yml`): Rohdaten als Parquet-Dateien im `warehouse`-Bucket
  ansehen.
- **Nessie-Historie**: `spark.sql("SELECT * FROM nessie.telemetrie.audio_spectrum.history").show()`
  zeigt alle Commits auf der Tabelle.
- **Branch aufräumen**: `spark.sql("DROP BRANCH \`analyse-schlecht-only\` IN nessie")`
- **Große Datenmengen**: Für sehr viele Zeilen lohnt ein echter JDBC-Connector
  (MariaDB Connector/J als zusätzliches Spark-Paket) statt des
  pandas-Zwischenschritts hier.
- **Persistenz**: Der Nessie-Katalog läuft mit `IN_MEMORY`-Store - ein
  Neustart des `nessie`-Containers verliert alle Branches/Commits (die
  Parquet-Dateien in MinIO selbst bleiben aber erhalten). Für Dauerbetrieb
  auf einen persistenten Version-Store umstellen, siehe
  [Nessie-Konfiguration](https://projectnessie.org/nessie-latest/configuration/).